# Web scraping bằng BeautifulSoup: Books to Scrape

**Mục tiêu:** tải HTML của một website, trích xuất dữ liệu có cấu trúc, sau đó báo cáo dữ liệu thu được và phương pháp sử dụng.

Website: [Books to Scrape](https://books.toscrape.com/) — sandbox được tạo để luyện scraping. Giá và rating được gán cho mục đích demo, không phản ánh giá hoặc chất lượng sách thực tế.

Notebook dùng **Requests + BeautifulSoup**.


## 1. Cài thư viện và cấu hình

BeautifulSoup đọc HTML, còn Requests gửi HTTP request. Sử dụng parser `html.parser` có sẵn của Python, không cần Selenium hoặc trình duyệt tự động vì dữ liệu nằm trong HTML trả về.

Nếu nâng cấp thư viện đã import, hãy restart kernel rồi chạy tiếp.


In [1]:
%pip install -q "requests>=2.31,<3" "beautifulsoup4>=4.12,<5" "pandas>=2.2.3,<4"


In [2]:
import json
import re
import time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path
from urllib.parse import urljoin, urlsplit
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

START_URL = "https://books.toscrape.com/catalogue/page-1.html"
DOMAIN = "books.toscrape.com"
ROBOTS_URL = "https://books.toscrape.com/robots.txt"
BOT_NAME = "LearningScraper"
MAX_PAGES = 3
DELAY_SECONDS = 1.0
TIMEOUT = (10, 30)  # connect timeout, read timeout (giây)
assert 1 <= MAX_PAGES <= 10, "Demo chỉ cho phép 1–10 trang."
assert DELAY_SECONDS >= 1

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_DIR = root / "books_scraping_outputs" / run_id
HTML_DIR = OUTPUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
print("Output:", OUTPUT_DIR)


Output: /content/books_scraping_outputs/20261028T033335_574979Z


## 2. Phạm vi và cách thu thập có trách nhiệm

- Chỉ gửi GET tới `books.toscrape.com`, không đăng nhập, vượt CAPTCHA hoặc truy cập dữ liệu cá nhân.
- Đọc `robots.txt` trước. Nếu bị cấm/không xác định được chính sách do lỗi mạng, dừng để kiểm tra. HTTP 404 nghĩa là không có file robots tại URL đó, không phải giấy phép chung để scrape website khác.
- Nghỉ ít nhất 1 giây giữa các trang; tôn trọng crawl-delay nếu được khai báo.
- Dừng khi có lỗi HTTP, phân trang lặp hoặc HTML không còn đúng cấu trúc. Không tự retry liên tục khi gặp 429/403.
- Tên sách, giá và rating được lấy từ trang danh sách. Không lấy mô tả đầy đủ, UPC, số lượng tồn kho hoặc nội dung trang chi tiết.

Quyền truy cập công khai và `robots.txt` không thay thế điều khoản sử dụng. Website được chọn có thông báo rõ rằng đây là demo dành cho scraping.


## 3. Phương pháp trích xuất

Mở DevTools/Inspect để xác định cấu trúc lặp. Mỗi sách nằm trong `article.product_pod`.

| Trường | Selector / nguồn | Cách lấy |
| --- | --- | --- |
| title | `h3 a` | Thuộc tính `title`, tránh tên bị rút gọn trên màn hình |
| price_gbp | `p.price_color` | Bỏ ký hiệu £, chuyển thành số |
| rating | `p.star-rating` | Đổi class One…Five thành 1…5 |
| availability | `p.availability` | Lấy text và gộp khoảng trắng |
| product_url | `h3 a` | Ghép href tương đối bằng urljoin |
| image_url | `.image_container img` | Chỉ lấy URL src, không tải ảnh |
| source_page, scraped_at_utc | HTTP request | Giữ nguồn và thời gian lấy |

Liên kết trang tiếp theo nằm trong `li.next a`. Đây là **trích xuất theo DOM/CSS selector**, không dùng regex để phân tích toàn bộ HTML; regex chỉ hỗ trợ kiểm tra định dạng giá. Xem [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).


In [3]:
def checked_url(base, link):
    url = urljoin(base, link)
    parts = urlsplit(url)
    if parts.scheme != "https" or parts.netloc != DOMAIN:
        raise ValueError("URL nằm ngoài phạm vi HTTPS/domain của demo.")
    return url

def parse_catalogue(html_bytes, page_url, scraped_at):
    soup = BeautifulSoup(html_bytes, "html.parser", from_encoding="utf-8")
    cards = soup.select("article.product_pod")
    if not cards:
        raise ValueError("Không thấy product_pod; dừng để kiểm tra cấu trúc HTML.")
    rows, errors = [], []
    for index, card in enumerate(cards, start=1):
        try:
            link = card.select_one("h3 a")
            price_node = card.select_one("p.price_color")
            if link is None or price_node is None:
                raise ValueError("Thiếu selector tên/giá.")
            title = link.get("title", "").strip()
            if not title:
                raise ValueError("Thiếu thuộc tính title đầy đủ.")
            price_match = re.fullmatch(r"£(\d+(?:\.\d{2})?)", price_node.get_text(strip=True))
            if price_match is None:
                raise ValueError("Định dạng giá GBP không đúng.")
            rating_node = card.select_one("p.star-rating")
            rating = next((RATING_MAP[c] for c in rating_node.get("class", []) if c in RATING_MAP), None) if rating_node else None
            availability_node = card.select_one("p.availability")
            image = card.select_one(".image_container img")
            rows.append({
                "title": title, "price_gbp": float(Decimal(price_match.group(1))),
                "currency": "GBP", "rating": rating,
                "availability": " ".join(availability_node.stripped_strings) if availability_node else None,
                "product_url": checked_url(page_url, link["href"]),
                "image_url": checked_url(page_url, image["src"]) if image and image.get("src") else None,
                "source_page": page_url, "scraped_at_utc": scraped_at,
            })
        except (ValueError, KeyError, TypeError) as error:
            errors.append({"source_page": page_url, "card_index": index, "error": str(error)})
    next_node = soup.select_one("li.next a")
    next_url = checked_url(page_url, next_node["href"]) if next_node else None
    return rows, errors, next_url, len(cards)

print("Đã định nghĩa parser; chưa tải trang ở cell này.")


Đã định nghĩa parser; chưa tải trang ở cell này.


## 4. Tải trang và đi theo phân trang

Cell này truy cập Internet thật. Mỗi trang thành công được lưu nguyên HTML để có thể kiểm tra/re-parse sau mà không tải lại.

Nếu một card thiếu tên/giá/URL hợp lệ, ghi vào log và bỏ card đó; các trường không bắt buộc bị thiếu được giữ trống. Nếu một trang không còn cấu trúc sách, dừng chứ không coi đó là trang rỗng hợp lệ.


In [4]:
rows, page_logs, extraction_errors = [], [], []
stop_reason = "not_started"
started_at = datetime.now(timezone.utc).isoformat()
crawl_started = time.perf_counter()

with requests.Session() as session:
    session.headers.update({"User-Agent": BOT_NAME + "/1.0 (educational scraping demo)"})
    robots_response = session.get(ROBOTS_URL, timeout=TIMEOUT, allow_redirects=False)
    robots_policy = {"url": ROBOTS_URL, "http_status": robots_response.status_code}
    robots = None
    if robots_response.status_code == 404:
        robots_policy["note"] = "robots.txt không tồn tại; website là sandbox cho scraping."
    else:
        robots_response.raise_for_status()
        if robots_response.status_code != 200 or "text/html" in robots_response.headers.get("Content-Type", "").lower():
            raise RuntimeError("Không đọc được robots.txt hợp lệ; dừng để kiểm tra.")
        robots = RobotFileParser()
        robots.parse(robots_response.text.splitlines())
        robots_policy["note"] = "Đã đọc robots.txt và kiểm tra từng URL."
    (OUTPUT_DIR / "robots_response.txt").write_text(robots_response.text, encoding="utf-8")
    crawl_delay = (robots.crawl_delay(BOT_NAME) or robots.crawl_delay("*") or 0) if robots else 0
    effective_delay = max(DELAY_SECONDS, float(crawl_delay))
    visited = set()
    url = START_URL

    for page_number in range(1, MAX_PAGES + 1):
        url = checked_url(START_URL, url)
        if url in visited:
            stop_reason = "pagination_loop"
            break
        if robots and not robots.can_fetch(BOT_NAME, url):
            stop_reason = "robots_disallowed"
            break
        time.sleep(effective_delay)
        visited.add(url)
        page_started = time.perf_counter()
        log = {"page_number": page_number, "url": url, "http_status": None}
        try:
            response = session.get(url, timeout=TIMEOUT, allow_redirects=False)
            log["http_status"] = response.status_code
            response.raise_for_status()
            if response.status_code != 200:
                raise ValueError("Không tự theo redirect hoặc response không phải HTTP 200.")
            if "text/html" not in response.headers.get("Content-Type", "").lower():
                raise ValueError("Response không phải HTML.")
            stamp = datetime.now(timezone.utc).isoformat()
            page_rows, page_errors, next_url, cards_seen = parse_catalogue(response.content, url, stamp)
            (HTML_DIR / f"page-{page_number:02d}.html").write_bytes(response.content)
            rows.extend(page_rows)
            extraction_errors.extend(page_errors)
            log.update(status="ok", cards_seen=cards_seen, extracted_rows=len(page_rows),
                       skipped_cards=len(page_errors))
        except (requests.RequestException, ValueError) as error:
            log.update(status="error", error=str(error))
            stop_reason = "page_error"
            next_url = None
        log["seconds"] = time.perf_counter() - page_started
        page_logs.append(log)
        print(f"Trang {page_number}: {log['status']}, HTTP {log['http_status']}, {log.get('extracted_rows', 0)} sách")
        if log["status"] != "ok":
            break
        if next_url is None:
            stop_reason = "no_next_page"
            break
        url = next_url
        stop_reason = "max_pages_reached"

crawl_seconds = time.perf_counter() - crawl_started
print("Lý do dừng:", stop_reason)
print("Số bản ghi trước khi loại trùng:", len(rows))


Trang 1: ok, HTTP 200, 20 sách
Trang 2: ok, HTTP 200, 20 sách
Trang 3: ok, HTTP 200, 20 sách
Lý do dừng: max_pages_reached
Số bản ghi trước khi loại trùng: 60


## 5. Kiểm tra và làm sạch dữ liệu

Mỗi URL sản phẩm đại diện cho một bản ghi. Giữ dòng đầu khi URL bị lặp, ghi rõ số dòng đã loại. Không tự điền rating hoặc giá trung bình cho trường bị thiếu.

Chỉ số rating là 1–5. Các kiểm tra dưới đây chỉ đánh giá định dạng và tính đầy đủ, không chứng minh dữ liệu đúng với thị trường thực tế.


In [5]:
columns = ["title", "price_gbp", "currency", "rating", "availability",
           "product_url", "image_url", "source_page", "scraped_at_utc"]
raw_df = pd.DataFrame(rows, columns=columns)
duplicate_count = int(raw_df.duplicated("product_url").sum())
df = raw_df.drop_duplicates("product_url").reset_index(drop=True)
df["price_gbp"] = pd.to_numeric(df["price_gbp"], errors="raise")
df["rating"] = pd.to_numeric(df["rating"], errors="raise").astype("Int64")
assert df["product_url"].is_unique
assert df["price_gbp"].dropna().ge(0).all()
assert df["rating"].dropna().between(1, 5).all()

quality_df = pd.DataFrame({
    "field": columns,
    "missing_count": [int(df[field].isna().sum()) for field in columns],
})
rating_df = df["rating"].value_counts().sort_index().rename_axis("rating").reset_index(name="book_count")
availability_df = df["availability"].value_counts(dropna=False).rename_axis("availability").reset_index(name="book_count")
display(df.head(10))
display(quality_df)
print("Dòng trùng bị loại:", duplicate_count)
print("Card bị bỏ do lỗi trích xuất:", len(extraction_errors))


,title,price_gbp,currency,rating,availability,product_url,image_url,source_page,scraped_at_utc
0,A Light in the Attic,51.77,GBP,3,In stock,https://books.toscrape.com/catalogue/a-light-i...,https://books.toscrape.com/media/cache/2c/da/2...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
1,Tipping the Velvet,53.74,GBP,1,In stock,https://books.toscrape.com/catalogue/tipping-t...,https://books.toscrape.com/media/cache/26/0c/2...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
2,Soumission,50.10,GBP,1,In stock,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/media/cache/3e/ef/3...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
3,Sharp Objects,47.82,GBP,4,In stock,https://books.toscrape.com/catalogue/sharp-obj...,https://books.toscrape.com/media/cache/32/51/3...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
4,Sapiens: A Brief History of Humankind,54.23,GBP,5,In stock,https://books.toscrape.com/catalogue/sapiens-a...,https://books.toscrape.com/media/cache/be/a5/b...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
5,The Requiem Red,22.65,GBP,1,In stock,https://books.toscrape.com/catalogue/the-requi...,https://books.toscrape.com/media/cache/68/33/6...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
6,The Dirty Little Secrets of Getting Your Dream...,33.34,GBP,4,In stock,https://books.toscrape.com/catalogue/the-dirty...,https://books.toscrape.com/media/cache/92/27/9...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
7,The Coming Woman: A Novel Based on the Life of...,17.93,GBP,3,In stock,https://books.toscrape.com/catalogue/the-comin...,https://books.toscrape.com/media/cache/3d/54/3...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
8,The Boys in the Boat: Nine Americans and Their...,22.60,GBP,4,In stock,https://books.toscrape.com/catalogue/the-boys-...,https://books.toscrape.com/media/cache/66/88/6...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00
9,The Black Maria,52.15,GBP,1,In stock,https://books.toscrape.com/catalogue/the-black...,https://books.toscrape.com/media/cache/58/46/5...,https://books.toscrape.com/catalogue/page-1.html,2026-10-28T03:33:37.055387+00:00


,field,missing_count
0,title,0
1,price_gbp,0
2,currency,0
3,rating,0
4,availability,0
5,product_url,0
6,image_url,0
7,source_page,0
8,scraped_at_utc,0


Dòng trùng bị loại: 0
Card bị bỏ do lỗi trích xuất: 0


## 6. Thống kê từ dữ liệu thực tế đã lấy

Phần này chỉ thống kê mẫu đã scrape, không đại diện toàn bộ website. Không suy ra “sách tốt nhất” từ rating hoặc đưa ra khuyến nghị mua sách vì đây là dữ liệu demo.


In [6]:
if df.empty:
    print("Chưa có sách hợp lệ để thống kê. Xem page_logs và lỗi phía trên.")
else:
    price_summary = df["price_gbp"].agg(["count", "min", "mean", "median", "max"])
    display(price_summary.rename("price_gbp").to_frame().round(2))
    display(rating_df)
    display(availability_df)
    display(df.nlargest(5, "price_gbp")[["title", "price_gbp", "rating"]])


,price_gbp
count,60.00
min,12.84
mean,35.00
median,33.48
max,57.31


,rating,book_count
0,1,15
1,2,8
2,3,13
3,4,10
4,5,14


,availability,book_count
0,In stock,60


,title,price_gbp,rating
40,Slow States of Collapse: Poems,57.31,3
15,Our Band Could Be Your Life: Scenes from the A...,57.25,3
58,The Past Never Ends,56.50,4
57,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
56,The Secret of Dreadwillow Carse,56.13,1


## 7. Xuất dữ liệu và tự tạo báo cáo

Báo cáo tổng hợp phương pháp, phạm vi, số trang, số bản ghi, lỗi, dữ liệu thiếu, phân bố rating và một số dòng dữ liệu thật. Nếu tải chưa đủ phạm vi hoặc có lỗi card, trạng thái được ghi rõ.

File HTML chỉ lưu để kiểm chứng, không thực thi script của website. CSV/JSON chứa metadata, không tải ảnh hoặc nội dung sách.


In [7]:
def markdown_table(frame):
    if frame.empty:
        return "Chưa có dữ liệu."
    def cell(value):
        return "—" if pd.isna(value) else str(value).replace("|", r"\|").replace("\n", " ")
    header = "| " + " | ".join(map(str, frame.columns)) + " |"
    divider = "| " + " | ".join(["---"] * len(frame.columns)) + " |"
    body = ["| " + " | ".join(cell(v) for v in row) + " |" for row in frame.itertuples(index=False, name=None)]
    return "\n".join([header, divider, *body])

successful_pages = sum(log["status"] == "ok" for log in page_logs)
scope_complete = stop_reason in {"max_pages_reached", "no_next_page"} and successful_pages > 0
status = "COMPLETED_SCOPE" if scope_complete and not extraction_errors else "PARTIAL_OR_ERRORS"
price_text = (
    f"Giá thấp nhất {df['price_gbp'].min():.2f}, trung bình {df['price_gbp'].mean():.2f}, "
    f"trung vị {df['price_gbp'].median():.2f}, cao nhất {df['price_gbp'].max():.2f} GBP."
    if not df.empty else "Không có dữ liệu hợp lệ để tính thống kê giá."
)
metadata = {
    "status": status, "data_source": "live_http", "started_at_utc": started_at,
    "start_url": START_URL, "max_pages": MAX_PAGES, "successful_pages": successful_pages,
    "rows_before_dedup": len(raw_df), "unique_rows": len(df),
    "duplicates_removed": duplicate_count, "skipped_cards": len(extraction_errors),
    "stop_reason": stop_reason, "crawl_seconds": crawl_seconds,
    "robots_policy": robots_policy, "delay_seconds": effective_delay,
    "method": "requests + BeautifulSoup(html.parser), CSS selectors, next-page links",
}
df.to_csv(OUTPUT_DIR / "books.csv", index=False, encoding="utf-8-sig")
df.to_json(OUTPUT_DIR / "books.json", orient="records", force_ascii=False, indent=2)
pd.DataFrame(page_logs).to_csv(OUTPUT_DIR / "page_log.csv", index=False, encoding="utf-8-sig")
(OUTPUT_DIR / "extraction_errors.json").write_text(json.dumps(extraction_errors, ensure_ascii=False, indent=2), encoding="utf-8")
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

report = f"""# Báo cáo web scraping — Books to Scrape

## 1. Website và phạm vi

- Website: [Books to Scrape](https://books.toscrape.com/).
- Thời điểm bắt đầu (UTC): {started_at}.
- Trạng thái: **{status}**. Lý do dừng: **{stop_reason}**.
- Giới hạn: {MAX_PAGES} trang danh sách; tải thành công {successful_pages} trang.
- Không truy cập trang chi tiết, không tải ảnh, không thu thập thông tin cá nhân.
- Đây là website demo; giá và rating không có ý nghĩa thị trường.

## 2. Phương pháp sử dụng

Dùng Requests gửi GET, timeout kết nối/đọc {TIMEOUT} giây, User-Agent học tập rõ ràng.
BeautifulSoup đọc HTML UTF-8 bằng html.parser; mỗi sách là article.product_pod.
Tên đầy đủ lấy từ thuộc tính title của h3 a. Giá lấy từ p.price_color, đổi GBP sang số.
Rating được lấy từ class One…Five của p.star-rating. Availability lấy text và gộp khoảng trắng.
URL tương đối được ghép bằng urljoin. Trang kế tiếp lấy từ li.next a, không hardcode toàn bộ danh sách trang.
Đọc robots.txt trước, nghỉ {effective_delay:.1f} giây giữa request trang; không tự theo redirect hoặc retry liên tục.

Robots HTTP {robots_policy['http_status']}: {robots_policy['note']}

## 3. Kết quả dữ liệu

- Số dòng trích xuất trước loại trùng: **{len(raw_df)}**.
- Số sản phẩm duy nhất: **{len(df)}**.
- Dòng trùng đã loại: {duplicate_count}; card bỏ do lỗi: {len(extraction_errors)}.
- Thời gian thu thập gồm robots và khoảng nghỉ: {crawl_seconds:.2f} giây.
- {price_text}

### Một số bản ghi thực tế

{markdown_table(df[['title', 'price_gbp', 'rating', 'availability']].head(5))}

### Phân bố rating trong mẫu

{markdown_table(rating_df)}

### Tình trạng hàng

{markdown_table(availability_df)}

### Kiểm tra dữ liệu thiếu

{markdown_table(quality_df)}

### Nhật ký từng trang

{markdown_table(pd.DataFrame(page_logs))}

## 4. Nhận xét và hạn chế

Chỉ thống kê phần dữ liệu đã lấy, không khẳng định đã thu thập toàn website.
Selector phụ thuộc cấu trúc DOM; khi website đổi cấu trúc cần kiểm tra và cập nhật parser.
Không có JavaScript rendering, nên cách này không đủ cho website chỉ nạp dữ liệu sau khi chạy JavaScript.
Rating đúng định dạng 1–5 không có nghĩa là đánh giá người dùng thật. Dữ liệu mẫu không dùng để khuyến nghị mua sách.
Tên sách lấy từ title thay vì text bị rút gọn; HTML gốc được lưu để đối chiếu.
Requests làm phần tải trang, BeautifulSoup làm phần phân tích HTML. Scrapy là framework khác;
Scapy là công cụ gói tin mạng, không được dùng trong bài này.

## 5. File đầu ra

books.csv, books.json, page_log.csv, extraction_errors.json, run_metadata.json,
robots_response.txt và thư mục html/ lưu trang đã tải thành công.

## 6. Nguồn tham khảo

- [Website và thông báo sandbox](https://books.toscrape.com/)
- [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Scrapy tutorial](https://docs.scrapy.org/en/latest/intro/tutorial.html)
- [Scapy introduction](https://scapy.readthedocs.io/en/latest/introduction.html)
"""
REPORT_PATH = OUTPUT_DIR / "scraping_report.md"
REPORT_PATH.write_text(report, encoding="utf-8")
display(Markdown(report))
print("Báo cáo:", REPORT_PATH)
print("CSV:", OUTPUT_DIR / "books.csv")


# Báo cáo web scraping — Books to Scrape

## 1. Website và phạm vi

- Website: [Books to Scrape](https://books.toscrape.com/).
- Thời điểm bắt đầu (UTC): 2026-10-28T03:33:35.607684+00:00.
- Trạng thái: **COMPLETED_SCOPE**. Lý do dừng: **max_pages_reached**.
- Giới hạn: 3 trang danh sách; tải thành công 3 trang.
- Không truy cập trang chi tiết, không tải ảnh, không thu thập thông tin cá nhân.
- Đây là website demo; giá và rating không có ý nghĩa thị trường.

## 2. Phương pháp sử dụng

Dùng Requests gửi GET, timeout kết nối/đọc (10, 30) giây, User-Agent học tập rõ ràng.
BeautifulSoup đọc HTML UTF-8 bằng html.parser; mỗi sách là article.product_pod.
Tên đầy đủ lấy từ thuộc tính title của h3 a. Giá lấy từ p.price_color, đổi GBP sang số.
Rating được lấy từ class One…Five của p.star-rating. Availability lấy text và gộp khoảng trắng.
URL tương đối được ghép bằng urljoin. Trang kế tiếp lấy từ li.next a, không hardcode toàn bộ danh sách trang.
Đọc robots.txt trước, nghỉ 1.0 giây giữa request trang; không tự theo redirect hoặc retry liên tục.

Robots HTTP 404: robots.txt không tồn tại; website là sandbox cho scraping.

## 3. Kết quả dữ liệu

- Số dòng trích xuất trước loại trùng: **60**.
- Số sản phẩm duy nhất: **60**.
- Dòng trùng đã loại: 0; card bỏ do lỗi: 0.
- Thời gian thu thập gồm robots và khoảng nghỉ: 4.79 giây.
- Giá thấp nhất 12.84, trung bình 35.00, trung vị 33.48, cao nhất 57.31 GBP.

### Một số bản ghi thực tế

| title | price_gbp | rating | availability |
| --- | --- | --- | --- |
| A Light in the Attic | 51.77 | 3 | In stock |
| Tipping the Velvet | 53.74 | 1 | In stock |
| Soumission | 50.1 | 1 | In stock |
| Sharp Objects | 47.82 | 4 | In stock |
| Sapiens: A Brief History of Humankind | 54.23 | 5 | In stock |

### Phân bố rating trong mẫu

| rating | book_count |
| --- | --- |
| 1 | 15 |
| 2 | 8 |
| 3 | 13 |
| 4 | 10 |
| 5 | 14 |

### Tình trạng hàng

| availability | book_count |
| --- | --- |
| In stock | 60 |

### Kiểm tra dữ liệu thiếu

| field | missing_count |
| --- | --- |
| title | 0 |
| price_gbp | 0 |
| currency | 0 |
| rating | 0 |
| availability | 0 |
| product_url | 0 |
| image_url | 0 |
| source_page | 0 |
| scraped_at_utc | 0 |

### Nhật ký từng trang

| page_number | url | http_status | status | cards_seen | extracted_rows | skipped_cards | seconds |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | https://books.toscrape.com/catalogue/page-1.html | 200 | ok | 20 | 20 | 0 | 0.931029286999987 |
| 2 | https://books.toscrape.com/catalogue/page-2.html | 200 | ok | 20 | 20 | 0 | 0.27292639600000257 |
| 3 | https://books.toscrape.com/catalogue/page-3.html | 200 | ok | 20 | 20 | 0 | 0.23466901199998347 |

## 4. Nhận xét và hạn chế

Chỉ thống kê phần dữ liệu đã lấy, không khẳng định đã thu thập toàn website.
Selector phụ thuộc cấu trúc DOM; khi website đổi cấu trúc cần kiểm tra và cập nhật parser.
Không có JavaScript rendering, nên cách này không đủ cho website chỉ nạp dữ liệu sau khi chạy JavaScript.
Rating đúng định dạng 1–5 không có nghĩa là đánh giá người dùng thật. Dữ liệu mẫu không dùng để khuyến nghị mua sách.
Tên sách lấy từ title thay vì text bị rút gọn; HTML gốc được lưu để đối chiếu.
Requests làm phần tải trang, BeautifulSoup làm phần phân tích HTML. Scrapy là framework khác;
Scapy là công cụ gói tin mạng, không được dùng trong bài này.

## 5. File đầu ra

books.csv, books.json, page_log.csv, extraction_errors.json, run_metadata.json,
robots_response.txt và thư mục html/ lưu trang đã tải thành công.

## 6. Nguồn tham khảo

- [Website và thông báo sandbox](https://books.toscrape.com/)
- [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Scrapy tutorial](https://docs.scrapy.org/en/latest/intro/tutorial.html)
- [Scapy introduction](https://scapy.readthedocs.io/en/latest/introduction.html)


Báo cáo: /content/books_scraping_outputs/20261028T033335_574979Z/scraping_report.md
CSV: /content/books_scraping_outputs/20261028T033335_574979Z/books.csv
